# LightLogger Data Processing Pipeline

## Overview

This notebook walks through the preprocessing stages implemented in `preprocessing_pipeline.py` for the LightLogger / FLIC dataset. The goal is to take raw world-camera and Neon eye-tracking recordings and turn them into validated, analysis-ready outputs for virtual foveation, SPD generation, and downstream summaries.

The workflow is intentionally staged. Early steps focus on transfer, download, and integrity checks; middle steps build synchronized world and Neon representations; later steps derive virtually foveated videos and SPDs. Running the notebook in order helps ensure that each stage receives the files and directory structure expected by the next one.

## Pipeline Stages

1. Acquire and verify raw data
2. Generate stitched world-camera videos from `GKA` chunk recordings
3. Verify Neon and world recordings are correctly paired
4. Run the egocentric mapper on Neon recordings and world videos
5. Tag task start/end windows and generate virtually foveated videos
6. Generate SPDs and aggregate summary outputs

Throughout the notebook, the markdown mirrors the responsibilities of the corresponding functions in `preprocessing_pipeline.py` so the processing logic and the documentation stay aligned.



## Environment Setup

Import the pipeline module and load the profiling extensions used for long-running preprocessing steps. Most cells in this notebook reload `preprocessing_pipeline` before execution so local code changes are picked up without restarting the kernel.



In [1]:
import importlib
import preprocessing_pipeline

In [ ]:
importlib.reload(preprocessing_pipeline)

raw_data_integrity_issues = preprocessing_pipeline.verify_data_integrity(verbose=True)


# Step 0 - Acquire and Verify Raw Recordings

## Purpose

Prepare the raw dataset so every requested subject/activity has the source material expected by downstream functions. In `preprocessing_pipeline.py`, this stage is handled by the transfer/download helpers plus the integrity checks that validate the required `GKA` and `Neon` directory contents.

## What We Check Here

- Light Logger world recordings are present in each activity's `GKA` folder
- Neon scene-video exports and time-series files are present in each activity's `Neon` folder
- Required Neon metadata and CSV tables exist before mapping or foveation begins
- The raw directory structure is consistent across the requested subjects and activities

## Expected Raw Layout

```text
FLIC_XXXX/
    activity_name/
        GKA/
        Neon/
```

Catching missing folders or incomplete downloads here is important because later stages assume these raw inputs already exist and are internally consistent.



### Transfer Light Logger World-Camera Recordings

Use `transfer_light_logger_recordings(...)` to copy the raw Light Logger recordings into the dataset root. This step is responsible for bringing the world-camera source material into the expected subject/activity structure before any video generation begins.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.transfer_light_logger_recordings(subjects_to_transfer=[1047],
                                                        overwrite_existing=False, 
                                                        verbose=True,
                                                        activities_to_transfer=["dark", "read", "chat", "walkIndoor", "walkOutdoor", "walkBiopond", "sitBiopond"])

### Download Pupil Labs Neon Exports

Use the Pupil Cloud download helper to retrieve the Neon `Timeseries Data + Scene Video` package for the same recordings. These exports provide the scene video and the gaze-related CSV files required by the later integrity, pairing, and egocentric-mapping steps.



#### Download the Neon `Timeseries Data + Scene Video` Packages

`download_pupil_cloud_recordings(...)` downloads the raw Neon export archives into the raw dataset directory. Keep this step early in the workflow so the subsequent verification cells can confirm that the expected `Neon` contents are present before any downstream processing starts.



In [ ]:
importlib.reload(preprocessing_pipeline)
api_key: str = "" # TODO: Fill this in, but NEVER commit this. If it is committed, immediately deactivate it on Pupil Cloud
preprocessing_pipeline.download_pupil_cloud_recordings(api_key, 
                                                       dst_dir="/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026", 
                                                       subjects_to_download=[1044], 
                                                       verbose=True,
                                                       overwrite_existing=False,
                                                       activities_to_download=["dark", "read", "chat", "walkIndoor", "walkOutdoor", "walkBiopond", "sitBiopond"]
                                                      )

Processing Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Downloading: 1044 | dark | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_1044/dark/Timeseries Data + Scene Video.zip


### Verify Raw `GKA` and `Neon` Contents

Run the integrity checks immediately after transfer/download. `verify_data_integrity(...)` checks that both modalities exist for each requested recording, while `verify_neon_integrity(...)` performs a more detailed pass over the extracted Neon contents such as `gaze.csv`, `fixations.csv`, `world_timestamps.csv`, and the scene video file.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_neon_integrity(verbose=True)

# Step 1 - Generate World-Camera Videos

## Purpose

Convert raw Light Logger chunk recordings stored in each activity's `GKA` folder into a single playable `W.avi` file per recording. This section corresponds to `generate_world_videos(...)` in `preprocessing_pipeline.py`.

## What This Step Does

- Finds every requested subject/activity with a valid raw `GKA` folder
- Reads the world-camera chunk files for that recording
- Optionally applies the preprocessing switches used by the pipeline, including color weighting, floor/ceiling correction, fielding correction, dark-noise removal, debayering, digital gain, and missing-frame filling
- Writes the stitched world video to the processing directory as `GKA/W.avi`

This world video is a required input for both the Neon/world pairing check and the egocentric mapper.



### Build `W.avi` Files from Raw `GKA` Chunks

Execute `generate_world_videos(...)` here after the raw data has been verified. Successful completion of this block means each selected recording now has a processed world-camera video available for synchronization and mapping.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_world_videos(overwrite_existing=True, 
                                             verbose=True, 
                                             apply_digital_gain=True, 
                                             apply_color_weights=True,
                                             debayer_images=True, 
                                             apply_floor_ceiling=True,
                                             remove_dark_noise=True, 
                                             apply_fielding_function=False, 
                                             subjects_to_process=[1038],
                                             activities_to_process=["walkOutdoor"]
                                            ) 

# Step 2 - Verify Neon and World Pairing

## Purpose

Confirm that each processed world video corresponds to the correct raw Neon recording before we run the egocentric mapper. In `preprocessing_pipeline.py`, `verify_world_neon_pairing(...)` loads the two modalities for the same subject/activity and helps confirm they belong to the same session.

## Why This Matters

A mismatched world/Neon pair can silently contaminate all downstream outputs. The pairing check is therefore one of the highest-value validation steps in the notebook. With `verbose=True`, the function displays frames that make it easier to visually confirm that the recordings match.



### Review World-Video and Neon Pairing

Run `verify_world_neon_pairing(...)` after world-video generation and before egocentric mapping. Treat any mismatch here as a blocker for the later synchronization-dependent steps.



In [ ]:
importlib.reload(preprocessing_pipeline)

# Return value is a dictionary of the recordings whose integrity was verified
videos_observed: dict[str, dict[str, bool]] = preprocessing_pipeline.verify_world_neon_pairing(verbose=True)

# Step 3 - Run the Egocentric Video Mapper

## Purpose

Generate the Neon-to-world alignment products used for gaze and fixation mapping. This section corresponds to `generate_egocentric_mapper_results(...)`, which uses the Neon recording directory plus the processed `W.avi` world video for each activity.

## Notes

- This step depends on the raw Neon export and the processed world video both being present
- The mapper can produce gaze mappings, fixation mappings, or both
- The notebook note about switching to the `egocentric_video_mapper` kernel still applies for this stage



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_egocentric_mapper_results(overwrite_existing=True, verbose=True, subjects_to_process=[28], activities_to_process=["walkBiopond"]) # TODO: There is some bug with 28 walkBiopond

# Step 4 - Generate Virtually Foveated Videos

## Purpose

Prepare the inputs needed for retinally centered video generation and then run the virtual foveation stage implemented by `generate_tag_task_start_ends(...)` and `generate_virtually_foveated_videos(...)`.

## What Happens in This Stage

1. Identify the task window for each recording
2. Use the tagged time bounds together with the mapped Neon outputs to drive virtual foveation
3. Optionally verify that the expected foveated outputs were produced

These outputs feed directly into the SPD-generation stage that follows later in the notebook.



### Tag Task Start and End Windows

Run `generate_tag_task_start_ends(...)` to locate the usable portion of each recording before virtual foveation begins. These tags define the time window that the MATLAB-based foveation step will use downstream.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_tag_task_start_ends(overwrite_existing=False, verbose=True)

### Build the Virtually Foveated Videos

Execute `generate_virtually_foveated_videos(...)` once the task windows have been tagged. After this step, the processing directory should contain the retinally centered video products that the SPD routines consume later in the pipeline.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_virtually_foveated_videos(overwrite_existing=True, verbose=True,  
                                                          activities_to_process=["chat", "walkBiopond"])

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_virtually_foveated_video_integrity(verbose=True, activities_to_skip=["phone", "lunch"])

## 5. Generate SPDs

### 1. Generate the Indinvidual SPDs and plots for all desired subjects and activitys 

In [ ]:
importlib.reload(preprocessing_pipeline)

color_modes: list[str] = ["a", "c_lm", "c_s"]
for color_mode in color_modes:
    preprocessing_pipeline.generate_spds(color_mode=color_mode,
                                        overwrite_existing=True, 
                                        verbose=True, 
                                        activities_to_process=["chat", "walkBiopond"], 
                                        )

In [ ]:
# Let's combine the SPDs for the desired colormodes 
importlib.reload(preprocessing_pipeline)

preprocessing_pipeline.combine_spds(overwrite_existing=True, 
                                    verbose=True, 
                                    activities_to_process=["chat", "walkBiopond"],
                                    color_modes_to_process=["a", "c_lm", "c_s"]
                                  )


### 2. Plot the indivudal SPDs (optionally if not done above)
We also have this as a separate optional step because the first step is ridiculously long in terms of runtime

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.adjust_spd_axes(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True)

#### 3. For each subject, output a grouped plot of their results for all activities


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_per_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 4. Generate average SPDs for each activity by averaging across subjects for that activity

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                    combine_figures=True, common_axes=True)

#### 5. Group the mean SPDs for each activity by the activity type


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_across_subjects(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 6. Generate average SPDs across all activities' averages

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_all(overwrite_existing=True, verbose=True, color_mode="L-M", activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True, common_axes=True)

In [ ]:
importlib.reload(preprocessing_pipeline)

sitting_vs_walking: dict = {"sitting": ["sitBiopond", "work", "chat"], 
                            "walking": ["walkBiopond", "walkIndoor", "walkOutdoor"]
                           }

indoor_vs_outdoor: dict = {"indoor": ["work", "chat", "walkIndoor"], 
                           "outdoor": ["walkOutdoor", "sitBiopond", "walkBiopond"]
                          }

preprocessing_pipeline.generate_spds_across_groups(overwrite_existing=True, 
                                                   color_mode="L-M",
                                                   verbose=True, 
                                                   activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                   combine_figures=True, 
                                                   common_axes=True,
                                                   groups=sitting_vs_walking
                                                )

### Generate actigraphy graphs

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_actigraphy_graphs(overwrite_exsiting=True, 
                                                  verbose=True, 
                                                  subjects_to_process=[2001],
                                                  activities_to_process=["walkBiopond"]
                                                )

In [ ]:
import matlab.engine
eng = matlab.engine.start_matlab()

print(eng.version(nargout=1))